In [96]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\gidia\.cache\kagglehub\datasets\arnabbiswas1\microsoft-azure-predictive-maintenance\versions\3


In [97]:
import pandas as pd

df = pd.read_csv(f'{path}\PdM_telemetry.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
print(df.info())
print(df.head())

<>:3: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<>:3: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
C:\Users\gidia\AppData\Local\Temp\ipykernel_20860\3032764308.py:3: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
  df = pd.read_csv(f'{path}\PdM_telemetry.csv')


<class 'pandas.DataFrame'>
RangeIndex: 876100 entries, 0 to 876099
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   datetime   876100 non-null  datetime64[us]
 1   machineID  876100 non-null  int64         
 2   volt       876100 non-null  float64       
 3   rotate     876100 non-null  float64       
 4   pressure   876100 non-null  float64       
 5   vibration  876100 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1)
memory usage: 40.1 MB
None
             datetime  machineID        volt      rotate    pressure  \
0 2015-01-01 06:00:00          1  176.217853  418.504078  113.077935   
1 2015-01-01 07:00:00          1  162.879223  402.747490   95.460525   
2 2015-01-01 08:00:00          1  170.989902  527.349825   75.237905   
3 2015-01-01 09:00:00          1  162.462833  346.149335  109.248561   
4 2015-01-01 10:00:00          1  157.610021  435.376873  111.886648   

   vibrat

In [98]:
print(df.describe())

                         datetime      machineID           volt  \
count                      876100  876100.000000  876100.000000   
mean   2015-07-02 17:59:59.999999      50.500000     170.777736   
min           2015-01-01 06:00:00       1.000000      97.333604   
25%           2015-04-02 12:00:00      25.750000     160.304927   
50%           2015-07-02 18:00:00      50.500000     170.607338   
75%           2015-10-02 00:00:00      75.250000     181.004493   
max           2016-01-01 06:00:00     100.000000     255.124717   
std                           NaN      28.866087      15.509114   

              rotate       pressure      vibration  
count  876100.000000  876100.000000  876100.000000  
mean      446.605119     100.858668      40.385007  
min       138.432075      51.237106      14.877054  
25%       412.305714      93.498181      36.777299  
50%       447.558150     100.425559      40.237247  
75%       482.176600     107.555231      43.784938  
max       695.020984     

In [99]:
from sklearn.preprocessing import StandardScaler

features = ['volt', 'rotate', 'pressure', 'vibration']

scaler_dict = {}

single_machine = df.set_index('datetime').groupby('machineID')

for i, df in single_machine:
    scaler = StandardScaler()
    scaler.fit(df[features])
    scaler_dict[i] = scaler

print(scaler_dict)

{1: StandardScaler(), 2: StandardScaler(), 3: StandardScaler(), 4: StandardScaler(), 5: StandardScaler(), 6: StandardScaler(), 7: StandardScaler(), 8: StandardScaler(), 9: StandardScaler(), 10: StandardScaler(), 11: StandardScaler(), 12: StandardScaler(), 13: StandardScaler(), 14: StandardScaler(), 15: StandardScaler(), 16: StandardScaler(), 17: StandardScaler(), 18: StandardScaler(), 19: StandardScaler(), 20: StandardScaler(), 21: StandardScaler(), 22: StandardScaler(), 23: StandardScaler(), 24: StandardScaler(), 25: StandardScaler(), 26: StandardScaler(), 27: StandardScaler(), 28: StandardScaler(), 29: StandardScaler(), 30: StandardScaler(), 31: StandardScaler(), 32: StandardScaler(), 33: StandardScaler(), 34: StandardScaler(), 35: StandardScaler(), 36: StandardScaler(), 37: StandardScaler(), 38: StandardScaler(), 39: StandardScaler(), 40: StandardScaler(), 41: StandardScaler(), 42: StandardScaler(), 43: StandardScaler(), 44: StandardScaler(), 45: StandardScaler(), 46: StandardScaler

In [100]:
scaled_frames = []

for i, df in single_machine:
    scaled = df.copy()
    scaled[features] = scaler_dict[i].transform(df[features])
    scaled_frames.append(scaled)

df_scaled = pd.concat(scaled_frames).sort_values(['machineID', 'datetime'])

print(df_scaled.describe())

           machineID          volt        rotate      pressure     vibration
count  876100.000000  8.761000e+05  8.761000e+05  8.761000e+05  8.761000e+05
mean       50.500000  1.737427e-16  4.853199e-17  9.479310e-17 -6.476880e-17
std        28.866087  1.000001e+00  1.000001e+00  1.000001e+00  1.000001e+00
min         1.000000 -4.720948e+00 -5.761340e+00 -4.546425e+00 -4.819702e+00
25%        25.750000 -6.754714e-01 -6.511285e-01 -6.665425e-01 -6.720890e-01
50%        50.500000 -1.106024e-02  1.818379e-02 -3.899188e-02 -2.736549e-02
75%        75.250000  6.595001e-01  6.754654e-01  6.071904e-01  6.342316e-01
max       100.000000  5.301436e+00  4.750478e+00  7.350799e+00  6.596818e+00


In [101]:
import numpy as np

def create_sequences(data, window_size):
    X = []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
    return np.array(X)

In [102]:
all_sequences = []

for i, df in df_scaled.groupby('machineID'):
    data = df[features].values
    sequences = create_sequences(data, window_size=24)
    all_sequences.append(sequences)

X = np.concatenate(all_sequences, axis=0)
print(X.shape)  # (samples, 24, 4)

(873700, 24, 4)


```py
epochs = 100

import torch                                      # framework deep learning principale
import torch.nn as nn                             # moduli per costruire reti neurali (layer, loss)
from torch.utils.data import DataLoader, TensorDataset  # gestione dataset e batch

# converte numpy array in tensori PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
dataset = TensorDataset(X_tensor, X_tensor)       # autoencoder: input = target (ricostruisce se stesso)
loader = DataLoader(dataset, batch_size=256, shuffle=True)  # serve i dati in batch durante il training

# classe che definisce l'architettura del modello
class LSTMAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()                         # inizializza la classe padre nn.Module
        # definisci i layer qui

    def forward(self, x):                          # forward pass: flusso dei dati attraverso il modello
        return x

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # ottimizzatore Adam, aggiorna i pesi
criterion = nn.MSELoss()                           # loss function: MSE tra input e ricostruzione

for epoch in range(epochs):
    for batch in loader:                           # itera sui batch
        optimizer.zero_grad()                      # azzera i gradienti del passo precedente
        output = model(batch)                      # forward pass
        loss = criterion(output, batch)            # calcola la loss
        loss.backward()                            # backpropagation: calcola i gradienti
        optimizer.step()                           # aggiorna i pesi
```

In [103]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        
        # ENCODER: comprime la sequenza da n_features → latent_dim
        # legge 24 timestep e produce un vettore compresso (hidden state finale)
        self.encoder = nn.LSTM(input_size=n_features, hidden_size=latent_dim, batch_first=True) #Numero feature, layer nascosto, (batch, timestep, feature)
        
        # DECODER: ricostruisce la sequenza da latent_dim → n_features
        # prende il vettore compresso e cerca di ricostruire i 24 timestep originali
        # se la ricostruzione è buona → comportamento normale
        # se la ricostruzione è cattiva (MSE alto) → comportamento anomalo
        self.decoder = nn.LSTM(input_size=latent_dim, hidden_size=n_features, batch_first=True)
        self.output_layer = nn.Linear(n_features, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden.squeeze(0)           # rimuove la prima dimensione → (batch, latent_dim)
        hidden = hidden.unsqueeze(1)         # aggiunge dimensione timestep → (batch, 1, latent_dim)
        hidden = hidden.repeat(1, 24, 1)     # ripete per 24 timestep → (batch, 24, latent_dim)
        output, _ = self.decoder(hidden)
        output = self.output_layer(output)  
        return output


In [104]:
model = LSTMAutoencoder(n_features=4, latent_dim=32)
print(model)

LSTMAutoencoder(
  (encoder): LSTM(4, 32, batch_first=True)
  (decoder): LSTM(32, 4, batch_first=True)
  (output_layer): Linear(in_features=4, out_features=4, bias=True)
)


In [105]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) 

model = model.to(device)  

cuda


In [106]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # aggiorna i pesi minimizzando la loss, lr=learning rate
criterion = nn.MSELoss()  # loss function: misura l'errore quadratico medio tra input e ricostruzione

In [107]:
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(X, dtype=torch.float32)          # converti numpy array in tensore PyTorch
dataset = TensorDataset(X_tensor, X_tensor)               # autoencoder: input = target
loader = DataLoader(dataset, batch_size=256, shuffle=True) # serve i dati in batch da 256

In [108]:
epochs = 50  # numero di volte che il modello vede tutto il dataset

for epoch in range(epochs):
    total_loss = 0
    for batch, target in loader:               # itera sui batch
        batch = batch.to(device)     # ← qui
        target = target.to(device)   # ← qui
        optimizer.zero_grad()                  # azzera i gradienti del passo precedente
        output = model(batch)                  # forward pass
        loss = criterion(output, target)       # calcola la loss
        loss.backward()                        # backpropagation
        optimizer.step()                       # aggiorna i pesi
        total_loss += loss.item()              # accumula la loss
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.6f}")

Epoch 1/50 - Loss: 0.877015
Epoch 2/50 - Loss: 0.863741
Epoch 3/50 - Loss: 0.843279
Epoch 4/50 - Loss: 0.831810
Epoch 5/50 - Loss: 0.826624
Epoch 6/50 - Loss: 0.824032
Epoch 7/50 - Loss: 0.822203
Epoch 8/50 - Loss: 0.820354
Epoch 9/50 - Loss: 0.815505
Epoch 10/50 - Loss: 0.812681
Epoch 11/50 - Loss: 0.810294
Epoch 12/50 - Loss: 0.808840
Epoch 13/50 - Loss: 0.807965
Epoch 14/50 - Loss: 0.806371
Epoch 15/50 - Loss: 0.805282
Epoch 16/50 - Loss: 0.804407
Epoch 17/50 - Loss: 0.803727
Epoch 18/50 - Loss: 0.802322
Epoch 19/50 - Loss: 0.800237
Epoch 20/50 - Loss: 0.798901
Epoch 21/50 - Loss: 0.798160
Epoch 22/50 - Loss: 0.797684
Epoch 23/50 - Loss: 0.797166
Epoch 24/50 - Loss: 0.796600
Epoch 25/50 - Loss: 0.796278
Epoch 26/50 - Loss: 0.795943
Epoch 27/50 - Loss: 0.795592
Epoch 28/50 - Loss: 0.795839
Epoch 29/50 - Loss: 0.795409
Epoch 30/50 - Loss: 0.795067
Epoch 31/50 - Loss: 0.795192
Epoch 32/50 - Loss: 0.794263
Epoch 33/50 - Loss: 0.794075
Epoch 34/50 - Loss: 0.793831
Epoch 35/50 - Loss: 0.7